<a href="https://colab.research.google.com/github/prajwal1-k/gen-ai/blob/pre-commit/Text_Summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install -q torch transformers sentencepiece

## Import the Packages

In [6]:
from transformers import pipeline
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [7]:
device = 0 if torch.cuda.is_available() else -1

print("Device:", "GPU" if device == 0 else "CPU")
print("Transformers version:", transformers.__version__)

Device: GPU
Transformers version: 5.20.2


## Taking the Pre-trained model (BART)
#### BART: Bidirectional and Auto-regressive transformers

In [8]:
model_name = "facebook/bart-large-cnn"

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

print("Model is loaded Successfully")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Model is loaded Successfully


## Create the text summarization function


## Text to Summarize

In [11]:
article = """ New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving court, Barrientos was arrested and charged with theft of service and criminal trespass for allegedly sneaking into the New York subway through an emergency exit, said Detective
Annette Markowski, a police spokeswoman. In total, Barrientos has been married 10 times, with nine of her marriages occurring between 1999 and 2002.
All occurred either in Westchester County, Long Island, New Jersey or the Bronx. She is believed to still be married to four men, and at one time, she was married to eight men at once, prosecutors say.
Prosecutors said the immigration scam involved some of her husbands, who filed for permanent residence status shortly after the marriages.
Any divorces happened only after such filings were approved. It was unclear whether any of the men will be prosecuted.
The case was referred to the Bronx District Attorney\'s Office by Immigration and Customs Enforcement and the Department of Homeland Security\'s
Investigation Division. Seven of the men are from so-called "red-flagged" countries, including Egypt, Turkey, Georgia, Pakistan and Mali.
Her eighth husband, Rashid Rajput, was deported in 2006 to his native Pakistan after an investigation by the Joint Terrorism Task Force.
If convicted, Barrientos faces up to four years in prison.  Her next court appearance is scheduled for May 18.
"""
print("Original Text: ", article)
print("Original words: ", article.split())

Original Text:   New York (CNN)When Liana Barrientos was 23 years old, she got married in Westchester County, New York.
A year later, she got married again in Westchester County, but to a different man and without divorcing her first husband.
Only 18 days after that marriage, she got hitched yet again. Then, Barrientos declared "I do" five more times, sometimes only within two weeks of each other.
In 2010, she married once more, this time in the Bronx. In an application for a marriage license, she stated it was her "first and only" marriage.
Barrientos, now 39, is facing two criminal counts of "offering a false instrument for filing in the first degree," referring to her false statements on the
2010 marriage license application, according to court documents.
Prosecutors said the marriages were part of an immigration scam.
On Friday, she pleaded not guilty at State Supreme Court in the Bronx, according to her attorney, Christopher Wright, who declined to comment further.
After leaving c

## Summarize the text

In [12]:
# Tokenize input
inputs = tokenizer(
    article,
    max_length=1024,
    truncation=True,
    return_tensors="pt"
).to(model.device)

# Generate summary
summary_ids = model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_length=100,
    min_length=30,
    num_beams=4,
    early_stopping=True
)

# Decode summary
summary = tokenizer.decode(
    summary_ids[0],
    skip_special_tokens=True
)

print("Generated Summary:")
print(summary)

Generated Summary:
Liana Barrientos, 39, is charged with two counts of offering a false instrument for filing in the first degree. In total, she has been married 10 times, with nine of her marriages occurring between 1999 and 2002. If convicted, she faces up to four years in prison.


## Advanced summarization function

In [15]:
def generate_summary(text, summary_type = "medium"):

  setting = {
      "short": (60,20),
      "medium": (120, 40),
      "long": (200,70)
  }
  men_len = setting[summary_type][0]
  min_len = setting[summary_type][1]

  inputs = tokenizer(
      text,
      max_length=1024,
      truncation=True,
      return_tensors="pt"
  ).to(model.device)

  with torch.no_grad():
    summary_ids = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=men_len,
        min_length=min_len,
        num_beams=4,    no_repeat_ngram_size=3,
            early_stopping=True
        )

    return tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True
    )

In [16]:
short_summary = generate_summary(article, "short")
medium_summary = generate_summary(article, "medium")
long_summary = generate_summary(article, "long")

print("Short Summary:")
print(short_summary)
print("\nMedium Summary:")
print(medium_summary)
print("\nLong Summary:")
print(long_summary)

Short Summary:
Liana Barrientos, 39, is charged with two counts of offering a false instrument for filing in the first degree. In total, she has been married 10 times, with nine of her marriages occurring between 1999 and 2002. She is believed to still be married to four men,

Medium Summary:
Liana Barrientos, 39, is charged with two counts of offering a false instrument for filing in the first degree. In total, she has been married 10 times, with nine of her marriages occurring between 1999 and 2002. If convicted, she faces up to four years in prison.

Long Summary:
Liana Barrientos, 39, is charged with two counts of offering a false instrument for filing in the first degree. In total, she has been married 10 times, with nine of her marriages occurring between 1999 and 2002. She is believed to still be married to four men, and at one time, she was married to eight men at once.
